In [16]:
import numpy as np
import os

# Program used to downscale the ground truth values without loosing any features
def downscale_depth_minpool(depth, target_h, target_w):
    H, W = depth.shape

    scale_y = H / target_h
    scale_x = W / target_w

    depth_ds = np.full((target_h, target_w), np.nan, dtype=np.float32)

    for y in range(target_h):
        for x in range(target_w):

            y0 = int(y * scale_y)
            y1 = max(y0 + 1, int((y + 1) * scale_y))

            x0 = int(x * scale_x)
            x1 = max(x0 + 1, int((x + 1) * scale_x))

            block = depth[y0:y1, x0:x1]

            valid = block[np.isfinite(block)]

            if valid.size > 0:
                depth_ds[y, x] = np.min(valid)

    return depth_ds

In [17]:
# Path to npy ground truth file
ground_truth_data = "/Users/devrathod/Documents/classes_2026/Senior Design/Eco-Cars-Depth-Estimation-2026/data/output/all_depths.npy"
# Path to model dpeth truth file
model_depth_data = "/Users/devrathod/Documents/classes_2026/Senior Design/Eco-Cars-Depth-Estimation-2026/data/monodepth_v2/all_depth.npy"

# Loading stacks 
groundtruth_stack = np.load(ground_truth_data)
model_stack = np.load(model_depth_data)

print("Ground truth shape:", groundtruth_stack.shape)
print("Model shape:", model_stack.shape)

if groundtruth_stack.shape[0] != model_stack.shape[0]:
    print("Error in the total number of frames")
else:
    print("Frame counts match:", groundtruth_stack.shape[0])

target_h = model_stack.shape[1]
target_w = model_stack.shape[2]

resized_gt = []

for frame in groundtruth_stack:
    resized = downscale_depth_minpool(frame, target_h, target_w)
    resized_gt.append(resized)

resized_gt = np.stack(resized_gt)

print("Downscaled GT shape:", resized_gt.shape)

np.save("/Users/devrathod/Documents/classes_2026/Senior Design/Eco-Cars-Depth-Estimation-2026/data/monodepth_v2/groundtruth.npy", resized_gt)

Ground truth shape: (199, 1280, 1920)
Model shape: (199, 192, 640)
Frame counts match: 199
Downscaled GT shape: (199, 192, 640)


In [18]:
"""
Create a boolean mask applying the Eigen crop to the input depth map.

Parameters:
    depth_map (np.ndarray): 2D array of depth values (H x W).

Returns:
    np.ndarray: Boolean mask with True inside the crop, False outside.
"""
def eigen_crop(depth_map):
    h, w = depth_map.shape
    crop_mask = np.zeros((h, w), dtype=bool)

    y1 = int(0.408 * h)
    y2 = int(0.992 * h)
    x1 = int(0.036 * w)
    x2 = int(0.964 * w)

    crop_mask[y1:y2, x1:x2] = True
    return crop_mask

In [11]:
# Path to npy ground truth file
ground_truth_data = "/Users/devrathod/Documents/classes_2026/Senior Design/Eco-Cars-Depth-Estimation-2026/data/monodepth_v2/groundtruth.npy"
# Path to model dpeth truth file
model_depth_data = "/Users/devrathod/Documents/classes_2026/Senior Design/Eco-Cars-Depth-Estimation-2026/data/monodepth_v2/all_depth.npy"

# Loading stacks 
groundtruth_stack = np.load(ground_truth_data)
model_stack = np.load(model_depth_data)

resized_gt = []
crop_masks = []

for frame in groundtruth_stack:
    # Resize GT frame to model resolution
    resized = downscale_depth_minpool(frame, target_h, target_w)
    resized_gt.append(resized)
    crop_masks.append(eigen_crop(resized))

resized_gt = np.stack(resized_gt)
crop_masks = np.stack(crop_masks)

# Flatten everything
gt_flat = resized_gt.flatten()
model_flat = model_stack.flatten()
mask_flat = crop_masks.flatten()

# Valid pixels: inside crop + no NaNs + depth > 0
valid_mask = (
    mask_flat &
    (~np.isnan(gt_flat)) & (gt_flat > 0) &
    (~np.isnan(model_flat)) & (model_flat > 0)
)

gt_valid = gt_flat[valid_mask]
model_valid = model_flat[valid_mask]

# Fit linear relation y = mx + c
m, c = np.polyfit(model_valid, gt_valid, 1)

# Predict GT from model
gt_pred = m * model_valid + c

# Compute RMSE
rmse = np.sqrt(np.mean((gt_valid - gt_pred) ** 2))

print(f"Scale (m): {m}")
print(f"Offset (c): {c}")
print(f"RMSE: {rmse}")

Scale (m): 27.367293935961985
Offset (c): 12.348771186196744
RMSE: 19.927658738191564


/var/folders/f2/t74vhry566j4hf8364yxtd_40000gn/T/ipykernel_98619/1263984240.py:38: RankWarning: Polyfit may be poorly conditioned
  m, c = np.polyfit(model_valid, gt_valid, 1)


In [19]:
import numpy as np

# Paths
ground_truth_data = "/Users/devrathod/Documents/classes_2026/Senior Design/Eco-Cars-Depth-Estimation-2026/data/monodepth_v2/groundtruth.npy"
model_depth_data = "/Users/devrathod/Documents/classes_2026/Senior Design/Eco-Cars-Depth-Estimation-2026/data/monodepth_v2/all_depth.npy"

# Load stacks
groundtruth_stack = np.load(ground_truth_data)
model_stack = np.load(model_depth_data)

print("GT shape:", groundtruth_stack.shape)
print("Model shape:", model_stack.shape)

# Apply Eigen crop masks
crop_masks = []
for frame in groundtruth_stack:
    crop_masks.append(eigen_crop(frame))

crop_masks = np.stack(crop_masks)

# Flatten
gt_flat = groundtruth_stack.flatten()
model_flat = model_stack.flatten()
mask_flat = crop_masks.flatten()

# Valid pixels
valid_mask = (
    mask_flat &
    np.isfinite(gt_flat) & (gt_flat > 0) &
    np.isfinite(model_flat) & (model_flat > 0)
)

gt_valid = gt_flat[valid_mask]
model_valid = model_flat[valid_mask]

print("Valid pixels:", len(gt_valid))

# ----------------------------
# Robust scale estimation
# ----------------------------
scale = np.median(gt_valid) / np.median(model_valid)

# Apply scaling
model_scaled = model_valid * scale

# ----------------------------
# Metrics
# ----------------------------
rmse = np.sqrt(np.mean((gt_valid - model_scaled) ** 2))

print(f"Scale factor: {scale}")
print(f"RMSE: {rmse}")

GT shape: (199, 192, 640)
Model shape: (199, 192, 640)
Valid pixels: 3533590
Scale factor: 72.6300048828125
RMSE: 28.599870681762695
